# Load RAG Eval Results
Loads `testingevals_a100` (results CSVs), `timingevals_a100/perfiii_*.csv`, and `timingevals_a100/trace_*.jsonl` into DataFrames with config columns parsed from filenames.

In [1]:
import os
import re
import json
import pandas as pd
from pathlib import Path

BASE = Path("/media/zman/extrahd2/randstuff")
TESTING_DIR  = BASE / "testingevals_a100"
TIMING_DIR   = BASE / "timingevals_a100"

In [2]:
pd.set_option('display.max_columns', None)


In [3]:
# Filename format (fields separated by ~):
# {model}~{dataset}~rag{use_rag}~k{topk}~comp{compress}~{compress_model}~r{compress_r}
# ~rerank{rerank}~n{rerank_k}~{rerank_model}~ret{index_model}~{text_data}~{index_data}
# ~batch{batch}~{batch_size}~pipe{pipeline}~dev{device_config}

CONFIG_KEYS = [
    "model", "dataset",
    "use_rag", "topk",
    "compress", "compress_model", "compress_r",
    "rerank", "rerank_k", "rerank_model",
    "index_model", "text_data", "index_data",
    "batch", "batch_size", "pipeline", "device_config",
]

def parse_config(stem: str) -> dict:
    """Parse RAG config from a filename stem (no prefix_, no extension)."""
    parts = stem.split("~")
    if len(parts) != len(CONFIG_KEYS):
        raise ValueError(f"Expected {len(CONFIG_KEYS)} fields, got {len(parts)}: {stem}")
    cfg = dict(zip(CONFIG_KEYS, parts))
    # Use removeprefix to strip exact leading tokens (not char-by-char like lstrip)
    cfg["use_rag"]       = int(cfg["use_rag"].removeprefix("rag"))
    cfg["topk"]          = int(cfg["topk"].removeprefix("k"))
    cfg["compress"]      = int(cfg["compress"].removeprefix("comp"))
    cfg["compress_r"]    = float(cfg["compress_r"].removeprefix("r"))
    cfg["rerank"]        = int(cfg["rerank"].removeprefix("rerank"))
    cfg["rerank_k"]      = int(cfg["rerank_k"].removeprefix("n"))
    cfg["index_model"]   = cfg["index_model"].removeprefix("ret")
    cfg["batch"]         = int(cfg["batch"].removeprefix("batch"))
    cfg["batch_size"]    = int(cfg["batch_size"])
    cfg["pipeline"]      = cfg["pipeline"].removeprefix("pipe")
    cfg["device_config"] = cfg["device_config"].removeprefix("dev")
    return cfg

def stem_from_path(path: Path, prefix: str) -> str:
    """Strip file prefix and extension to get the config stem."""
    name = path.stem  # e.g. results_Llama-3.2-1B~...
    return name[len(prefix) + 1:]  # +1 for the underscore

# Quick sanity check
sample = next(TESTING_DIR.glob("results_*.csv"))
print(parse_config(stem_from_path(sample, "results")))

{'model': 'Llama-3.2-3B-Instruct', 'dataset': 'nq_dataset', 'use_rag': 0, 'topk': 0, 'compress': 0, 'compress_model': 'none', 'compress_r': 1.0, 'rerank': 0, 'rerank_k': 0, 'rerank_model': 'none', 'index_model': 'e5-base-v2', 'text_data': 'test_sample_2018_sent', 'index_data': 'e5_Flat_index', 'batch': 0, 'batch_size': 1, 'pipeline': 'standard', 'device_config': '187789'}


## 1. Testing Results (`testingevals_a100/results_*.csv`)
Columns: `gold, prediction, rouge, f1, em, recall` + config columns

In [ ]:
result_frames = []
for f in sorted(TESTING_DIR.glob("results_*.csv")):
    cfg = parse_config(stem_from_path(f, "results"))
    df = pd.read_csv(f)
    df.insert(0, "idx", range(len(df)))
    for k, v in cfg.items():
        df[k] = v
    result_frames.append(df)

df_results = pd.concat(result_frames, ignore_index=True)
print(f"df_results: {df_results.shape}")
df_results.head()

## 2. Timing / Performance (`timingevals_a100/perfiii_*.csv`)
Each row = one query run; many timing/GPU columns + config columns.

In [ ]:
import ast

def expand_dict_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Expand dict-valued columns (string or real dicts) into flat columns, recursively."""
    while True:
        to_drop, new_cols = [], {}
        for col in df.columns:
            sample = df[col].dropna()
            if sample.empty:
                continue
            v = sample.iloc[0]
            # Handle both string-serialized dicts and actual dict objects
            if isinstance(v, str) and v.strip().startswith('{'):
                try:
                    parsed = df[col].map(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
                except Exception:
                    continue
            elif isinstance(v, dict):
                parsed = df[col]
            else:
                continue
            try:
                keys = list(parsed.dropna().iloc[0].keys())
                for k in keys:
                    suffix = f"gpu{k}" if isinstance(k, int) else k
                    new_cols[f"{col}_{suffix}"] = parsed.map(
                        lambda d, k=k: d.get(k) if isinstance(d, dict) else None
                    )
                to_drop.append(col)
            except Exception:
                pass
        if not to_drop:
            break
        df = df.drop(columns=to_drop)
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df


perf_frames = []
for f in sorted(TIMING_DIR.glob("perfiii_*.csv")):
    cfg = parse_config(stem_from_path(f, "perfiii"))
    df = pd.read_csv(f)
    df = expand_dict_cols(df)
    for k, v in cfg.items():
        df[k] = v
    perf_frames.append(df)

df_perf = pd.concat(perf_frames, ignore_index=True)
print(f"df_perf: {df_perf.shape}")
df_perf.head()

## 3. Traces (`timingevals_a100/trace_*.jsonl`)
Each line = one query.  
- `meta`: question, mode, idx  
- `trace_stage_durations`: per-stage timing (e.g. `generate`)  
- `samples`: time-series GPU/CPU power — aggregated here to mean/peak per GPU

In [6]:
trace_rows = []

for f in sorted(TIMING_DIR.glob("trace_*.jsonl")):
    cfg = parse_config(stem_from_path(f, "trace"))
    with open(f) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            row = dict(cfg)

            # Meta fields
            meta = rec.get("meta", {})
            row["question"] = meta.get("question")
            row["mode"]     = meta.get("mode")
            row["idx"]      = meta.get("idx")

            # Markers (e.g. generate_start, generate_end timestamps)
            for marker, t in (rec.get("markers") or {}).items():
                row[f"marker_{marker}"] = t

            # Stage durations
            for stage, dur in (rec.get("trace_stage_durations") or {}).items():
                row[f"dur_{stage}"] = dur

            # Aggregate all sample sensor columns: mean and peak
            samples = rec.get("samples") or []
            if samples:
                sdf = pd.DataFrame(samples).drop(columns=["t"], errors="ignore")
                for col in sdf.columns:
                    row[f"{col}_mean"] = sdf[col].mean()
                    row[f"{col}_peak"] = sdf[col].max()

            trace_rows.append(row)

df_trace = pd.DataFrame(trace_rows)
print(f"df_trace: {df_trace.shape}")
df_trace.head()

df_trace: (540, 57)


,model,dataset,use_rag,topk,compress,compress_model,compress_r,rerank,rerank_k,rerank_model,index_model,text_data,index_data,batch,batch_size,pipeline,device_config,question,mode,idx,marker_generate_start,marker_generate_end,dur_generate,gpu0_power_w_mean,gpu0_power_w_peak,gpu0_sm_pct_mean,gpu0_sm_pct_peak,gpu0_mem_pct_mean,gpu0_mem_pct_peak,gpu0_mem_mb_mean,gpu0_mem_mb_peak,gpu1_power_w_mean,gpu1_power_w_peak,gpu1_sm_pct_mean,gpu1_sm_pct_peak,gpu1_mem_pct_mean,gpu1_mem_pct_peak,gpu1_mem_mb_mean,gpu1_mem_mb_peak,gpu2_power_w_mean,gpu2_power_w_peak,gpu2_sm_pct_mean,gpu2_sm_pct_peak,gpu2_mem_pct_mean,gpu2_mem_pct_peak,gpu2_mem_mb_mean,gpu2_mem_mb_peak,gpu3_power_w_mean,gpu3_power_w_peak,gpu3_sm_pct_mean,gpu3_sm_pct_peak,gpu3_mem_pct_mean,gpu3_mem_pct_peak,gpu3_mem_mb_mean,gpu3_mem_mb_peak,cpu_pct_mean,cpu_pct_peak
0,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,"Who else besides ""The Golden Jet"" was born in ...",single,0,0.009243,0.763727,0.754484,67.092591,68.385,0.890511,2,0.000000,0,3979.406478,3999.1875,69.570511,69.655,0.0,0,0.0,0,1049.1875,1049.1875,65.675226,65.706,0.0,0,0.0,0,1049.1875,1049.1875,57.298766,57.347,0.0,0,0.0,0,1049.1875,1049.1875,91.664964,380.5
1,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,"What is the first name of the woman who, like ...",single,1,0.001613,0.099967,0.098353,78.556579,84.490,8.631579,16,2.368421,5,3999.187500,3999.1875,69.655000,69.655,0.0,0,0.0,0,1049.1875,1049.1875,65.638000,65.638,0.0,0,0.0,0,1049.1875,1049.1875,57.325263,57.347,0.0,0,0.0,0,1049.1875,1049.1875,57.231579,188.1
2,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,Is Karjiang a mountain and Sherpi Kangri a mou...,single,2,0.001633,0.259521,0.257888,84.317184,94.452,19.591837,24,5.897959,7,3999.187500,3999.1875,69.473143,69.655,0.0,0,0.0,0,1049.1875,1049.1875,65.643551,65.706,0.0,0,0.0,0,1049.1875,1049.1875,57.288000,57.288,0.0,0,0.0,0,1049.1875,1049.1875,100.259184,372.5
3,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,How many visitors per year does the first stop...,single,3,0.001734,0.064975,0.063240,87.572615,87.976,27.692308,40,8.615385,14,3999.187500,3999.1875,69.627308,69.655,0.0,0,0.0,0,1049.1875,1049.1875,65.690308,65.706,0.0,0,0.0,0,1049.1875,1049.1875,57.288000,57.288,0.0,0,0.0,0,1049.1875,1049.1875,100.400000,188.3
4,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,What kind of ideology is Torment's drummer inv...,single,4,0.001714,0.079741,0.078027,87.401500,88.300,40.000000,40,14.000000,14,3999.187500,3999.1875,69.583000,69.583,0.0,0,0.0,0,1049.1875,1049.1875,65.638000,65.638,0.0,0,0.0,0,1049.1875,1049.1875,57.288000,57.288,0.0,0,0.0,0,1049.1875,1049.1875,93.756250,188.9


## Summary

In [7]:
print("df_results columns:", list(df_results.columns))
print()
print("df_perf columns:",    list(df_perf.columns))
print()
print("df_trace columns:",   list(df_trace.columns))

df_results columns: ['gold', 'prediction', 'rouge', 'f1', 'em', 'recall', 'model', 'dataset', 'use_rag', 'topk', 'compress', 'compress_model', 'compress_r', 'rerank', 'rerank_k', 'rerank_model', 'index_model', 'text_data', 'index_data', 'batch', 'batch_size', 'pipeline', 'device_config']

df_perf columns: ['rag_total_timer', 'rag_total', 'ttft', 'decode_time', 'generate_total_time', 'generate', 'tokens_generated', 'tokens_per_second', 'rag_total_timer_iter_0', 'rag_total_iter_0', 'ttft_iter_0', 'decode_time_iter_0', 'generate_total_time_iter_0', 'generate_iter_0', 'tokens_generated_iter_0', 'tokens_per_second_iter_0', 'question', 'prediction', 'mode', 'idx', 'cuda_visible_devices', 'num_gpus', 'gpu_0_name', 'gpu_1_name', 'gpu_2_name', 'gpu_3_name', 'init_cpu_start', 'load_embedder_cpu_before', 'load_embedder_cpu_after', 'embedder_device', 'load_generator_cpu_before', 'load_generator_cpu_after', 'generator_device', 'rerank', 'rerank_model', 'rerank_top_n', 'faiss_nvml_after_gpu_0_used_m

In [8]:
for name, df in [("df_results", df_results), ("df_perf", df_perf), ("df_trace", df_trace)]:
    with open(BASE / f"{name}_columns.txt", "w") as f:
        f.write("\n".join(df.columns))
    print(f"Wrote {name}_columns.txt ({len(df.columns)} columns)")

Wrote df_results_columns.txt (23 columns)
Wrote df_perf_columns.txt (135 columns)
Wrote df_trace_columns.txt (57 columns)


In [9]:
df_perf.head()

,rag_total_timer,rag_total,ttft,decode_time,generate_total_time,generate,tokens_generated,tokens_per_second,rag_total_timer_iter_0,rag_total_iter_0,ttft_iter_0,decode_time_iter_0,generate_total_time_iter_0,generate_iter_0,tokens_generated_iter_0,tokens_per_second_iter_0,question,prediction,mode,idx,cuda_visible_devices,num_gpus,gpu_0_name,gpu_1_name,gpu_2_name,gpu_3_name,init_cpu_start,load_embedder_cpu_before,load_embedder_cpu_after,embedder_device,load_generator_cpu_before,load_generator_cpu_after,generator_device,rerank,rerank_model,rerank_top_n,faiss_nvml_after_gpu_0_used_mb,faiss_nvml_after_gpu_0_free_mb,faiss_nvml_after_gpu_1_used_mb,faiss_nvml_after_gpu_1_free_mb,faiss_nvml_after_gpu_2_used_mb,faiss_nvml_after_gpu_2_free_mb,faiss_nvml_after_gpu_3_used_mb,faiss_nvml_after_gpu_3_free_mb,gen_gpu_before_allocated,gen_gpu_before_peak,gen_gpu_all_before_gpu_0_allocated,gen_gpu_all_before_gpu_0_peak,gen_gpu_all_before_gpu_1_allocated,gen_gpu_all_before_gpu_1_peak,gen_gpu_all_before_gpu_2_allocated,gen_gpu_all_before_gpu_2_peak,gen_gpu_all_before_gpu_3_allocated,gen_gpu_all_before_gpu_3_peak,gen_gpu_all_after_gpu_0_allocated,gen_gpu_all_after_gpu_0_peak,gen_gpu_all_after_gpu_1_allocated,gen_gpu_all_after_gpu_1_peak,gen_gpu_all_after_gpu_2_allocated,gen_gpu_all_after_gpu_2_peak,gen_gpu_all_after_gpu_3_allocated,gen_gpu_all_after_gpu_3_peak,gen_gpu_after_allocated,gen_gpu_after_peak,faiss_nvml_after_gpu_0_iter_0_used_mb,faiss_nvml_after_gpu_0_iter_0_free_mb,faiss_nvml_after_gpu_1_iter_0_used_mb,faiss_nvml_after_gpu_1_iter_0_free_mb,faiss_nvml_after_gpu_2_iter_0_used_mb,faiss_nvml_after_gpu_2_iter_0_free_mb,faiss_nvml_after_gpu_3_iter_0_used_mb,faiss_nvml_after_gpu_3_iter_0_free_mb,gen_gpu_before_iter_0_allocated,gen_gpu_before_iter_0_peak,gen_gpu_all_before_gpu_0_iter_0_allocated,gen_gpu_all_before_gpu_0_iter_0_peak,gen_gpu_all_before_gpu_1_iter_0_allocated,gen_gpu_all_before_gpu_1_iter_0_peak,gen_gpu_all_before_gpu_2_iter_0_allocated,gen_gpu_all_before_gpu_2_iter_0_peak,gen_gpu_all_before_gpu_3_iter_0_allocated,gen_gpu_all_before_gpu_3_iter_0_peak,gen_gpu_all_after_gpu_0_iter_0_allocated,gen_gpu_all_after_gpu_0_iter_0_peak,gen_gpu_all_after_gpu_1_iter_0_allocated,gen_gpu_all_after_gpu_1_iter_0_peak,gen_gpu_all_after_gpu_2_iter_0_allocated,gen_gpu_all_after_gpu_2_iter_0_peak,gen_gpu_all_after_gpu_3_iter_0_allocated,gen_gpu_all_after_gpu_3_iter_0_peak,gen_gpu_after_iter_0_allocated,gen_gpu_after_iter_0_peak,energy_j_gpu0,energy_j_gpu1,energy_j_gpu2,energy_j_gpu3,power_mean_w_gpu0,power_mean_w_gpu1,power_mean_w_gpu2,power_mean_w_gpu3,util_mean_gpu0,util_mean_gpu1,util_mean_gpu2,util_mean_gpu3,util_peak_gpu0,util_peak_gpu1,util_peak_gpu2,util_peak_gpu3,init_gpu_start_gpu_0_allocated,init_gpu_start_gpu_0_peak,init_gpu_start_gpu_1_allocated,init_gpu_start_gpu_1_peak,init_gpu_start_gpu_2_allocated,init_gpu_start_gpu_2_peak,init_gpu_start_gpu_3_allocated,init_gpu_start_gpu_3_peak,load_embedder_gpu_after_allocated,load_embedder_gpu_after_peak,load_generator_gpu_after_allocated,load_generator_gpu_after_peak,model,dataset,use_rag,topk,compress,compress_model,compress_r,rerank_k,index_model,text_data,index_data,batch,batch_size,pipeline,device_config
0,1.779247e+09,0.764543,0.674824,0.045547,0.720371,0.720371,5,109.777843,1.779247e+09,0.764543,0.674824,0.045547,0.720371,0.720371,5,109.777843,"Who else besides ""The Golden Jet"" was born in ...",- Michael Caine,single,0,"0,1,2,3",4,NVIDIA A100-SXM4-40GB,NVIDIA A100-SXM4-40GB,NVIDIA A100-SXM4-40GB,NVIDIA A100-SXM4-40GB,840.160156,1241.070312,1375.453125,cuda:0,1375.453125,1474.0,cuda:0,0,none,3,3965.1875,36994.8125,1049.1875,39910.8125,1049.1875,39910.8125,1049.1875,39910.8125,2911.747584,2911.747584,2911.747584,2911.747584,0.0,0.0,0.0,0.0,0.0,0.0,2920.267264,2944.360960,0.0,0.0,0.0,0.0,0.0,0.0,2920.267264,2944.360960,3965.1875,36994.8125,1049.1875,39910.8125,1049.1875,39910.8125,1049.1875,39910.8125,2911.747584,2911.747584,2911.747584,2911.747584,0.0,0.0,0.0,0.0,0.0,0.0,2920.267264,2944.3609